# Pre processing with Pytorch

For this brief preprocessing participation activity, we import the california housing dataset and apply a neural network with PyTorch to predict the housing prices.

The work consists of implementing a preprocessing pipeline, including outlier removal, standard scaling, and the omission of specific columns

In [48]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

We utilize the fetch function provided by sklearn to import our dataset.

In [49]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


We then proceed to split our data into training, validation, and testing datasets.

We define our two groups of columns, the ones that are going to pass through the preprocessing layers and the ones being omitted.

After that, we calculate the 25th and 75th percentiles from only our main preprocessing columns. After that, we calculate the IQR value and define our lower and upper bounds. This was only applied to our training data to avoid data leakage. We finalize this section by applying a mask, which essentially removes values falling outside of our bounds, effectuating outlier removal.

To conclude, we calculate the mean and std of our training data, again to avoid data leakage. This was done for using the standard scaling layer.

Note: Claude LLM was utilized to create lines 4 and 5 of this code cell and avoid manually defining the remaining columns.

In [50]:
X_train, X_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_testval, y_testval, test_size=0.5, random_state=42)

lat_lon_cols = ['Latitude', 'Longitude']
other_cols = [c for c in X_train.columns if c not in lat_lon_cols]

Q1 = X_train[other_cols].quantile(0.25)
Q3 = X_train[other_cols].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

mask = ((X_train[other_cols] >= lower_bound) & (X_train[other_cols] <= upper_bound)).all(axis=1)
X_train = X_train[mask]
y_train = y_train[mask]

mean = torch.tensor(X_train[other_cols].mean().values, dtype=torch.float32)
std = torch.tensor(X_train[other_cols].std().values, dtype=torch.float32)

Next is the the neural networks scaling layer, which takes the mean and std of the training data and calculates the new value.

In [51]:
class ScalingLayer(nn.Module):
    def __init__(self, mean: torch.Tensor, std: torch.Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return (X - self.mean) / (self.std + 1e-8)

This preprocessing layer was then implemented in its own general preprocessing layer, which will be the one actually applied in our main neural network class. This class, apart from setting the scaler, is also in charge of applying the scaler to the rows to which we want it applied (all except latitude and longitude) and passing the remaining ones without any filter.

In [52]:
class PreprocessingLayer(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.scaler = ScalingLayer(mean, std)

    def forward(self, X_main, X_rest):
        return torch.cat([
            self.scaler(X_main),
            X_rest
        ], dim=1)

Next is the main NN class, this serves as our first model, using 3 hidden layers, and applying the preprocessing before entering those layers.

In [53]:
class PreProcessingNN(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, hidden3_size, output_size, mean, std):
        super().__init__()
        
        self.preprocessing = PreprocessingLayer(mean, std)

        self.hidden1 = nn.Linear(input_size, hidden1_size)
        self.hidden2 = nn.Linear(hidden1_size, hidden2_size)
        self.hidden3 = nn.Linear(hidden2_size, hidden3_size)

        self.output = nn.Linear(hidden3_size, output_size)

        self.relu = nn.ReLU()

    def forward(self, X_main, X_rest):
        x = self.preprocessing(X_main, X_rest)
        
        x = self.hidden1(x)
        x = self.relu(x)

        x = self.hidden2(x)
        x = self.relu(x)

        x = self.hidden3(x)
        x = self.relu(x)
        
        x = self.output(x)
        return x.squeeze()
    


Next cell is in charge of assigning assigning computation to the GPU and creating both our model and optimizer. This first model consists of 3 hidden layers of size 40 each.

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PreProcessingNN(input_size=8, hidden1_size=40, hidden2_size=40, hidden3_size=40, output_size=1, mean=mean, std=std)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

device

device(type='cuda')

We set our loss function as the mean squared error.

In [55]:
loss_fn = nn.MSELoss()

We then set our dataset to give to our dataloader, this is where we actually split our main train data into the main rows, those who will pass through the preprocessing layers, and the rest, which will pass normally as they are.

In [56]:
X_train_main = torch.tensor(X_train[other_cols].values, dtype=torch.float32)
X_train_rest = torch.tensor(X_train[lat_lon_cols].values, dtype=torch.float32)
y_tensor = torch.tensor(y_train.values, dtype=torch.float32)

dataset = TensorDataset(X_train_main, X_train_rest, y_tensor)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

We define our training function

In [57]:
def train_batch(model, X_main, X_rest, y_batch, optimizer, loss_fn):
    predictions = model(X_main, X_rest)
    loss = loss_fn(predictions, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss

And we then proceed to apply the same separation for our validation data, as well as creating our custom function that manually computes the MSE of the model.

In [58]:
X_val_main = torch.tensor(X_val[other_cols].values, dtype=torch.float32).to(device)
X_val_rest = torch.tensor(X_val[lat_lon_cols].values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)

def computeModelMSE(model, X_main, X_rest, y):
    model.eval()

    with torch.no_grad():
        predictions = model(X_main, X_rest)
        squared = torch.pow(y - predictions, 2)
        _sum = torch.sum(squared)
        total = y.size(0)

    return _sum / total

We finalize this model's implementation by training it

In [59]:
model.train()

num_epochs = 30
for epoch in range(num_epochs):
    for X_main, X_rest, y_batch in dataloader:
        X_main = X_main.to(device)
        X_rest = X_rest.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model, X_main, X_rest, y_batch, optimizer, loss_fn)

    mse = computeModelMSE(model, X_val_main, X_val_rest, y_val)
    model.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation MSE: {mse:.2f}")

Epoch 1, Loss: 0.9091, Validation MSE: 0.74
Epoch 2, Loss: 0.2239, Validation MSE: 0.79
Epoch 3, Loss: 0.5675, Validation MSE: 0.83
Epoch 4, Loss: 0.3181, Validation MSE: 0.87
Epoch 5, Loss: 0.8476, Validation MSE: 0.97
Epoch 6, Loss: 0.5389, Validation MSE: 1.01
Epoch 7, Loss: 0.3727, Validation MSE: 1.20
Epoch 8, Loss: 0.3021, Validation MSE: 1.25
Epoch 9, Loss: 0.5988, Validation MSE: 1.39
Epoch 10, Loss: 0.2539, Validation MSE: 1.43
Epoch 11, Loss: 0.5007, Validation MSE: 1.72
Epoch 12, Loss: 0.6473, Validation MSE: 1.77
Epoch 13, Loss: 0.5453, Validation MSE: 1.90
Epoch 14, Loss: 0.2184, Validation MSE: 1.92
Epoch 15, Loss: 0.2081, Validation MSE: 2.02
Epoch 16, Loss: 0.1997, Validation MSE: 2.29
Epoch 17, Loss: 0.4558, Validation MSE: 2.53
Epoch 18, Loss: 0.2083, Validation MSE: 2.34
Epoch 19, Loss: 0.5213, Validation MSE: 2.39
Epoch 20, Loss: 0.3025, Validation MSE: 2.03
Epoch 21, Loss: 0.4472, Validation MSE: 2.63
Epoch 22, Loss: 0.2254, Validation MSE: 2.31
Epoch 23, Loss: 0.3

And by printing the model's actual testing performance

In [60]:
X_test_main = torch.tensor(X_test[other_cols].values, dtype=torch.float32).to(device)
X_test_rest = torch.tensor(X_test[lat_lon_cols].values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

mse1 = computeModelMSE(model, X_test_main, X_test_rest, y_test)

print(f"Testing performance - MSE: {mse1:.2f}")

Testing performance - MSE: 0.96


## Additional models evaluation

### Model 2 - 3 hidden layers of 30 neurons each, trained for 20 epochs

In [61]:
model2 = PreProcessingNN(input_size=8, hidden1_size=30, hidden2_size=30, hidden3_size=30, output_size=1, mean=mean, std=std)
model2.to(device)

optimizer2 = optim.Adam(model2.parameters(), lr=0.001)

model2.train()

num_epochs = 20
for epoch in range(num_epochs):
    for X_main, X_rest, y_batch in dataloader:
        X_main = X_main.to(device)
        X_rest = X_rest.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model2, X_main, X_rest, y_batch, optimizer2, loss_fn)

    mse = computeModelMSE(model2, X_val_main, X_val_rest, y_val)
    model2.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation MSE: {mse:.2f}")

mse2 = computeModelMSE(model2, X_test_main, X_test_rest, y_test)

print(f"Testing performance - MSE: {mse2:.2f}")

Epoch 1, Loss: 0.5498, Validation MSE: 0.80
Epoch 2, Loss: 0.2957, Validation MSE: 1.00
Epoch 3, Loss: 0.4484, Validation MSE: 0.95
Epoch 4, Loss: 0.3944, Validation MSE: 1.35
Epoch 5, Loss: 0.4843, Validation MSE: 1.59
Epoch 6, Loss: 0.5768, Validation MSE: 1.53
Epoch 7, Loss: 0.1935, Validation MSE: 1.57
Epoch 8, Loss: 0.4077, Validation MSE: 1.66
Epoch 9, Loss: 0.4155, Validation MSE: 1.61
Epoch 10, Loss: 0.3607, Validation MSE: 1.48
Epoch 11, Loss: 0.3701, Validation MSE: 1.48
Epoch 12, Loss: 0.3616, Validation MSE: 1.57
Epoch 13, Loss: 0.1782, Validation MSE: 1.41
Epoch 14, Loss: 0.5082, Validation MSE: 1.43
Epoch 15, Loss: 0.3271, Validation MSE: 1.33
Epoch 16, Loss: 0.2972, Validation MSE: 1.29
Epoch 17, Loss: 0.1984, Validation MSE: 1.22
Epoch 18, Loss: 0.7556, Validation MSE: 0.99
Epoch 19, Loss: 0.4281, Validation MSE: 1.17
Epoch 20, Loss: 0.4248, Validation MSE: 0.94
Testing performance - MSE: 0.60


### Model 3 - 3 hidden layers of 50, 40, and 30 neurons, trained for 25 epochs

In [62]:
model3 = PreProcessingNN(input_size=8, hidden1_size=50, hidden2_size=40, hidden3_size=30, output_size=1, mean=mean, std=std)
model3.to(device)

optimizer3 = optim.Adam(model3.parameters(), lr=0.001)

model3.train()

num_epochs = 25
for epoch in range(num_epochs):
    for X_main, X_rest, y_batch in dataloader:
        X_main = X_main.to(device)
        X_rest = X_rest.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model3, X_main, X_rest, y_batch, optimizer3, loss_fn)

    mse = computeModelMSE(model3, X_val_main, X_val_rest, y_val)
    model3.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation MSE: {mse:.2f}")

mse3 = computeModelMSE(model3, X_test_main, X_test_rest, y_test)

print(f"Testing performance - MSE: {mse3:.2f}")

Epoch 1, Loss: 0.3356, Validation MSE: 0.84
Epoch 2, Loss: 1.4598, Validation MSE: 1.26
Epoch 3, Loss: 0.5179, Validation MSE: 1.45
Epoch 4, Loss: 0.4378, Validation MSE: 1.58
Epoch 5, Loss: 0.6844, Validation MSE: 1.81
Epoch 6, Loss: 0.6789, Validation MSE: 2.16
Epoch 7, Loss: 0.3332, Validation MSE: 2.43
Epoch 8, Loss: 0.3927, Validation MSE: 2.73
Epoch 9, Loss: 0.2520, Validation MSE: 3.32
Epoch 10, Loss: 0.1849, Validation MSE: 3.70
Epoch 11, Loss: 0.2827, Validation MSE: 4.22
Epoch 12, Loss: 0.4075, Validation MSE: 4.40
Epoch 13, Loss: 0.4939, Validation MSE: 4.42
Epoch 14, Loss: 0.1607, Validation MSE: 3.64
Epoch 15, Loss: 0.6541, Validation MSE: 4.06
Epoch 16, Loss: 0.3797, Validation MSE: 4.60
Epoch 17, Loss: 0.7721, Validation MSE: 4.13
Epoch 18, Loss: 0.4576, Validation MSE: 4.38
Epoch 19, Loss: 0.5479, Validation MSE: 4.44
Epoch 20, Loss: 0.3510, Validation MSE: 4.58
Epoch 21, Loss: 0.2074, Validation MSE: 5.34
Epoch 22, Loss: 0.6315, Validation MSE: 4.75
Epoch 23, Loss: 0.2

### Model 4 - 3 hidden layers of 80 neurons each, trained for 35 epochs

In [63]:
model4 = PreProcessingNN(input_size=8, hidden1_size=80, hidden2_size=80, hidden3_size=80, output_size=1, mean=mean, std=std)
model4.to(device)

optimizer4 = optim.Adam(model4.parameters(), lr=0.001)

model4.train()

num_epochs = 35
for epoch in range(num_epochs):
    for X_main, X_rest, y_batch in dataloader:
        X_main = X_main.to(device)
        X_rest = X_rest.to(device)
        y_batch = y_batch.to(device)

        loss = train_batch(model4, X_main, X_rest, y_batch, optimizer4, loss_fn)

    mse = computeModelMSE(model4, X_val_main, X_val_rest, y_val)
    model4.train()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Validation MSE: {mse:.2f}")

mse4 = computeModelMSE(model4, X_test_main, X_test_rest, y_test)

print(f"Testing performance - MSE: {mse4:.2f}")

Epoch 1, Loss: 0.7667, Validation MSE: 0.64
Epoch 2, Loss: 0.3188, Validation MSE: 0.95
Epoch 3, Loss: 0.6381, Validation MSE: 1.00
Epoch 4, Loss: 0.2596, Validation MSE: 1.29
Epoch 5, Loss: 0.4885, Validation MSE: 1.51
Epoch 6, Loss: 0.2860, Validation MSE: 1.71
Epoch 7, Loss: 0.3269, Validation MSE: 2.06
Epoch 8, Loss: 0.3566, Validation MSE: 1.96
Epoch 9, Loss: 0.6886, Validation MSE: 1.91
Epoch 10, Loss: 0.5611, Validation MSE: 2.00
Epoch 11, Loss: 0.1735, Validation MSE: 2.06
Epoch 12, Loss: 0.4477, Validation MSE: 1.55
Epoch 13, Loss: 0.4352, Validation MSE: 1.44
Epoch 14, Loss: 0.2651, Validation MSE: 1.47
Epoch 15, Loss: 0.1825, Validation MSE: 1.20
Epoch 16, Loss: 0.1330, Validation MSE: 1.31
Epoch 17, Loss: 0.4251, Validation MSE: 1.46
Epoch 18, Loss: 0.3968, Validation MSE: 1.54
Epoch 19, Loss: 0.3845, Validation MSE: 1.21
Epoch 20, Loss: 0.2874, Validation MSE: 1.45
Epoch 21, Loss: 0.2338, Validation MSE: 1.28
Epoch 22, Loss: 0.2848, Validation MSE: 1.48
Epoch 23, Loss: 0.5

Based on the 4 models' performance, we can declare that the best one was the second one, which was simpler than the rest. Overall, we got very high MSE values for both the validation subsets and test subsets. This is because we removed the outliers from the training data but kept them in the remaining data to ensure that no data leakage was occurring. This implementation of the use of IQR is good if outliers are rare, but because they are a normal occurrence in this specific dataset, a more valid approach would be to cap the values, retaining the same quantity of data without training on outlying values.

This approach was chosen to align with the assignment's requirement of outlier removal rather than capping.